In [158]:
import pandas as pd
from pathlib import Path

# 추천 시스템의 평가
- 과거 행동으로 학습 -> 아직 안 본 후보들 중 추천
- 실제로 나중에 좋아한 아이템을 맞히는지 평가

우선 과거 데이터로 학습을 한다음에 나중에 좋아한 아이템을 맞히는지 평가하는 방식

> 최종적으로 사용자가 좋아했던 데이터를 이용해서 사용자가 앞으로 좋아할 데이터를 맞추는 게임이다.

In [159]:
DATA_DIR = Path().resolve().parents[1] / "data"
DATA_DIR.resolve(), DATA_DIR.exists()

(WindowsPath('C:/LANG_CHAIN_2026/2026-05-19_KDT_lang_chain/02_self_study/machine_learning/ranking/data'),
 True)

In [160]:
rankings = pd.read_csv(
  DATA_DIR / "ml-100k" / "u.data",
  sep="\t",
  names=["user_id", "movie_id", "rating", "timestamp"]
)

rankings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [161]:
rankings[["user_id", "movie_id"]].min(), rankings[["user_id", "movie_id"]].max()

(user_id     1
 movie_id    1
 dtype: int64,
 user_id      943
 movie_id    1682
 dtype: int64)

In [162]:
rankings[["user_id", "movie_id"]] = rankings[["user_id", "movie_id"]] -1

In [163]:
rankings.head()

,user_id,movie_id,rating,timestamp
0,195,241,3,881250949
1,185,301,3,891717742
2,21,376,1,878887116
3,243,50,2,880606923
4,165,345,1,886397596


In [164]:
rankings.shape, rankings.dtypes

((100000, 4),
 user_id      int64
 movie_id     int64
 rating       int64
 timestamp    int64
 dtype: object)

In [165]:
# 사용자별로 순서 정렬해주기
rankings = rankings.sort_values(
  ["user_id", "timestamp"],
  ascending=[True, False]
)

rankings[:5], rankings[500:505]

(       user_id  movie_id  rating  timestamp
 3248         0        73       1  889751736
 19699        0       101       2  889751736
 30479        0       255       4  889751712
 47638        0         4       3  889751712
 687          0       170       5  889751711,
        user_id  movie_id  rating  timestamp
 28363        4       167       3  875636691
 33920        4       413       3  875636691
 4921         4       172       4  875636675
 26872        4       207       4  875636675
 42966        4       203       4  875636675)

In [166]:
# 사용자별 4-5 데이터들 모두 꺼내서 저장해주기
positive_choice = rankings[rankings["rating"] >= 4]

positive_choice.head()

,user_id,movie_id,rating,timestamp
30479,0,255,4,889751712
687,0,170,5,889751711
88259,0,110,5,889751711
10922,0,241,5,889751633
25255,0,31,5,888732909


In [167]:
# 긍정적인 평가중에서 일부만 뽑아 테스트 데이터셋으로 설정해주기. 일단 회원당 긍정적인 평가 개수 확인
positive_choice.groupby("user_id").count().mean(), positive_choice.groupby("user_id").count().std()

(movie_id     58.784501
 rating       58.784501
 timestamp    58.784501
 dtype: float64,
 movie_id     54.696664
 rating       54.696664
 timestamp    54.696664
 dtype: float64)

In [168]:
# 사용자별로 자신이 선택한 영화에 대해서 n개 뽑아주기
test_dataset = positive_choice.groupby("user_id").head(2)
test_dataset.head(10)

,user_id,movie_id,rating,timestamp
30479,0,255,4,889751712
687,0,170,5,889751711
12150,1,315,5,888979693
54744,1,299,4,888979197
9021,2,317,4,889237482
29583,2,319,5,889237482
48826,3,10,4,892004520
12151,3,293,5,892004409
2686,4,23,4,879198229
8482,4,39,4,879198109


In [169]:
positive_choice = positive_choice.drop(test_dataset.index)
positive_choice[positive_choice["user_id"] == 0].tail(5)

,user_id,movie_id,rating,timestamp
22971,0,165,5,874965677
48214,0,155,4,874965556
74577,0,164,5,874965518
59972,0,167,5,874965478
92487,0,171,5,874965478


In [170]:
# 사용자가 본 movie_id df를 만들어주기
user_seen = rankings.groupby("user_id")["movie_id"].apply(set)
type(user_seen), user_seen.index[:5], user_seen[:5]

(pandas.Series,
 Index([0, 1, 2, 3, 4], dtype='int64', name='user_id'),
 user_id
 0    {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
 1    {256, 0, 257, 285, 306, 9, 12, 13, 268, 271, 2...
 2    {257, 259, 263, 267, 270, 271, 287, 293, 298, ...
 3    {257, 259, 263, 10, 270, 287, 293, 299, 300, 3...
 4    {0, 1, 16, 20, 23, 24, 28, 39, 41, 49, 61, 62,...
 Name: movie_id, dtype: object)

In [171]:
movie_set = set(rankings["movie_id"])
len(movie_set)

1682

In [193]:
# 하나의 평가에 대해서 3개의 부정적인 선택 만들기
import random
import torch

num_negative = 3

users = []
items = []
labels = []

for _, row in positive_choice.iterrows():
  
  # positive 하나 추가
  users.append(row["user_id"])
  items.append(row["movie_id"])
  labels.append(1.0)

  # # negative n개 추가
  candidates = list(
    user_seen[row["user_id"]] 
    -  # 사용자가 시청한 영화중에서 긍정적인 평가 제외해주기
    set(positive_choice[positive_choice["user_id"] == row["user_id"]]["movie_id"])
    -
    set(test_dataset[test_dataset["user_id"] == row["user_id"]]["movie_id"])
  )
  random.shuffle(candidates)

  num = min(num_negative, len(candidates))

  users.extend([row["user_id"] for _ in range(num)])
  items.extend(candidates[:num])
  labels.extend([0.0 for _ in range(num)])

users = torch.tensor(users, dtype=torch.long)
items = torch.tensor(items, dtype=torch.long)
labels = torch.tensor(labels, dtype=torch.float32)

print("users:", len(users))
print("items:", len(items))
print("labels:", len(labels))

users: 213378
items: 213378
labels: 213378


In [174]:
print("users:", len(users))
print("items:", len(items))
print("labels:", len(labels))

users: 213910
items: 213910
labels: 213910


In [175]:
len(rankings)

100000

In [176]:
len(positive_choice)

53491

In [177]:
len(users), len(items), len(labels)

(213910, 213910, 213910)

In [192]:
from torch import nn

# 모델 만들어보기
class SimpleRecommender(nn.Module):
  def __init__(self, num_users, num_items, embedding_dim=32):
    super().__init__()
    
    self.user_embedding = nn.Embedding(num_users, embedding_dim)
    self.item_embedding = nn.Embedding(num_items, embedding_dim)

    nn.init.normal_(self.user_embedding.weight, mean=0, std=0.05)
    nn.init.normal_(self.item_embedding.weight, mean=0, std=0.05)

  def forward(self, user_ids, item_ids):
    users_vec = self.user_embedding(user_ids)
    items_vec = self.item_embedding(item_ids)

    return (users_vec * items_vec).sum(dim=1)

In [179]:
num_users = rankings["user_id"].nunique()
num_items = rankings["movie_id"].nunique()

num_users, num_items

(943, 1682)

In [194]:
import torch
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(users, items, labels)

loader = DataLoader(
  dataset,
  batch_size=1024,
  shuffle=True
)

model = SimpleRecommender(num_users, num_items)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

In [195]:
EPOCH = 50

for epoch in range(1, EPOCH+1):
  total_loss = 0

  for batch_users, batch_items, batch_labels in loader:
    optimizer.zero_grad()

    logits = model(users, items)

    loss: torch.Tensor = criterion(logits, labels)

    loss.backward()

    optimizer.step()

    total_loss += loss.item()

  if epoch % 10 == 0:
    print(f"epoch: {epoch} | loss: {total_loss / len(loader)}")

epoch: 10 | loss: 0.05413232005811764
epoch: 20 | loss: 0.006062266677528905
epoch: 30 | loss: 0.0010954501663232178
epoch: 40 | loss: 0.0002865433629154357
epoch: 50 | loss: 8.675115382545026e-05


In [196]:
negative_pairs = set(
  zip(
    users[labels==0].tolist(),
    items[labels==0].tolist()
  )
)

test_pairs = set(
  zip(
    test_dataset["user_id"],
    test_dataset["movie_id"]
  )
)

print(
  "test in negative",
  len(test_pairs & negative_pairs)
)

test in negative 0


In [197]:
import numpy as np

# 테스트 켜주기
model.eval()

with torch.inference_mode():
  user_id = 0

  # 사용자가 실제로 좋아하는 아이템(영화) 꺼내주기
  test_items = set(
    test_dataset[
      test_dataset["user_id"] == user_id
    ]["movie_id"]
  )

  # 사용자의 전체 평가한 영화 꺼내주기
  user_data = rankings[rankings["user_id"] == user_id]

  # movie_id 뽑아주기 (사용자가 본적 없는) + 테스트 데이터
  candidates = (
    movie_set - set(user_data["movie_id"])
  ) | test_items
  candidates = list(candidates)

  # user_id 리스트
  user_tensor = torch.full(
    (len(candidates),),
    user_id,
    dtype=torch.long
  )

  # movie_id set을 tensor (n, )로 바꿔주기
  item_tensor = torch.tensor(
    candidates,
    dtype=torch.long
  )

  # 모두 예측하기
  scores = model(
    torch.tensor(user_id, dtype=torch.long),
    torch.tensor(candidates, dtype=torch.long)
  )

  # DataFrame로 변경하기
  result = pd.DataFrame({
    "item_id": candidates,
    "score": scores.detach().cpu().numpy()
  }, dtype=np.float32)

  # 이미 train에 쓴건 빼주기
  result = result.sort_values(["score"], ascending=[False])
  print(result)

  

model.train()

     item_id      score
434    704.0  95.863319
802   1072.0  83.166389
227    497.0  78.134972
278    548.0  77.917221
389    659.0  67.269676
..       ...        ...
30     300.0 -56.675911
608    878.0 -62.251472
29     299.0 -63.816414
23     293.0 -64.829948
200    470.0 -78.640396

[1412 rows x 2 columns]


SimpleRecommender(
  (user_embedding): Embedding(943, 32)
  (item_embedding): Embedding(1682, 32)
)

In [199]:
len(movie_set)

1682

In [198]:

print(test_items)
result["rank"] = range(1, len(result)+1)
result[result["item_id"].isin(list(test_items))]

{170, 255}


,item_id,score,rank
0,170.0,54.987808,14
1,255.0,22.313721,120
